# Data Retrieval

#### Objective
This notebook retrieves the historical stock market data. We are using the **Yahoo Finance API** to pull data for three specific ETFs:
* **SPY** (S&P 500 ETF) - Top 500 US companies
* **QQQ** (NASDAQ ETF) - Top 100 NASDAQ (tech-heavy) companies
* **VTI** (Total Market ETF) - The entire US stock market

We are extracting the maximum available historical data. The raw files are saved to `data/raw/` with no modifications

In [ ]:
import yfinance as yf
import pandas as pd
import os

RAW_DIR = "../data/raw" # Define the path to our raw data folder (relative to this notebook)
etfs = ["SPY", "QQQ", "VTI"] 

os.makedirs(RAW_DIR, exist_ok=True) # Ensure the raw directory exists

## Download and Save Raw ETF Data

### What gets saved:
One CSV per ETF in `data/raw/` 
— `SPY_raw.csv`, `QQQ_raw.csv`, `VTI_raw.csv`  

In [ ]:
for ticker in etfs:
    print(f"Downloading {ticker} data.")
    
    # Fetch data from Yahoo Finance API
    df = yf.download(
        ticker,
        period="max", # Get all available historical data
        auto_adjust=False, # Prevent library from dropping the 'Adj Close' column
        multi_level_index=False,  # Ensures a flat DataFrame for easy column renaming
    )
    
    df.reset_index(inplace=True)
    
    # Rename 'Adj Close' to 'Adjusted Close'
    if 'Adj Close' in df.columns:
        df.rename(columns={'Adj Close': 'Adjusted Close'}, inplace=True)
    
    df["Ticker"] = ticker # Add a 'Ticker' column to identify the data 
    
    # Define the exact column orderfor final CSV output
    column_order = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adjusted Close', 'Volume']
    df = df[column_order]
    
    # Save to the raw directory as CSV files
    output_path = os.path.join(RAW_DIR, f"{ticker}_raw.csv")
    df.to_csv(output_path, index=False)
    
    print(f"Saved {len(df)} rows → {output_path}") # Print a success message with the no. of rows and the file path
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}\n") # Print a success message with the date range captured

## Sanity Check & Validation
We verify that our raw fileshave been created successfully and contain the correct columns.

In [ ]:
for ticker in etfs:
    path = os.path.join(RAW_DIR, f"{ticker}_raw.csv")
    df   = pd.read_csv(path)

    print(f"{ticker}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Nulls:\n{df.isnull().sum()}") # Check for any missing/null values in the raw data
    
    print("\nPreview:")
    print(df.head(10))